# Drosophila Chemical–Gene (STITCH) — Relation-Wise KG Triple Construction

## Purpose

This notebook processes **Chemical–Protein interaction data** from the STITCH database for *Drosophila melanogaster* and transforms it into standardized Chemical–Gene relation-wise Knowledge Graph (KG) triples. FlyBase protein IDs are mapped to gene symbols via gProfiler, and chemicals are annotated with PubChem IUPAC names.

## Pipeline Overview

| Step | Section | Description |
|------|---------|-------------|
| 1 | gProfiler Mapping | Load gProfiler (FlyBase Protein ID → Gene Symbol + Description) |
| 2 | PubChem Mapping | Load PubChem (CID → IUPAC Name) |
| 3 | STITCH Loading & Processing | Load raw STITCH TSV, clean IDs, map proteins to genes, annotate |
| 4 | Filter, Validate, Deduplicate & Export | Remove unmapped, validate, deduplicate, export |

## Final Output Schema

| Column | Description |
|--------|-------------|
| `head` | PubChem CID (chemical identifier) |
| `relation` | `ChemicalEntity_Gene` |
| `tail` | Gene symbol (FlyBase) |
| `head_type` | `ChemicalEntity` |
| `relation_type` | (NaN — no sub-relation) |
| `tail_type` | `Gene` |
| `kg_source` | `STITCH` |
| `head_id_is` | `Pubchem` |
| `tail_id_is` | `Flybase` |
| `head_detail_name` | PubChem IUPAC chemical name |
| `tail_detail_name` | Gene description from gProfiler |
| `species` | `D.melanogaster` |

## Data Download Instructions

All databases used in this notebook must be downloaded prior to execution.
Please refer to the **central download instructions document** for detailed steps:

📄 **[Download Instructions — Link to be added]**

### Required Files

| File | Source | Description |
|------|--------|-------------|
| `7227.protein_chemical.links.detailed.v5.0.tsv` | STITCH | Raw chemical–protein interaction file for Drosophila (tax_id: 7227) |
| `Gprofiler_FBpp_2_Gene_Symbol.csv` | gProfiler | FlyBase protein ID to gene symbol/description mapping |
| `combined_df.pkl` | PubChem (pre-processed) | PubChem compound data with CID and IUPAC names |

---
## Setup

Import required libraries and define all base paths.

In [47]:
import pandas as pd
import numpy as np
!pwd

/storage/Arushi/090526_EvoAge/kg_formation/data_processing/species_specific_data_source/Drosophila


In [46]:
# =============================================================================
# BASE PATHS — Update these to match your local directory structure
# =============================================================================
your_path_here = '/storage/Arushi/090526_EvoAge/kg_formation/data_collection/'

# gProfiler mapping file (FlyBase protein ID → gene symbol)
GPROFILER_PATH = f'{your_path_here}flybase'
GPROFILER_PATH = f'{your_path_here}flybase/Gprofiler_FBpp_2_Gene_Symbol.csv'


# NCBI gene info for C. elegans

NCBI_GENE_INFO_PATH = f'{your_path_here}databases_for_mapping/ncbi/Drosophila_melanogaster.gene_info'

# PubChem compound data (pre-processed pickle with CID, IUPAC names)
PUBCHEM_PATH = f'{your_path_here}databases_for_mapping/pubchem/combined_df.pkl'

# Raw STITCH chemical-protein interaction file for Drosophila
STITCH_RAW_PATH = f'{your_path_here}stitch/dmel/7227.protein_chemical.links.detailed.v5.0.tsv'

# Final output path
OUT_PATH = '/storage/Arushi/090526_EvoAge/kg_formation/processed_data/stitch/'

OUTPUT_PATH = '/storage/Arushi/090526_EvoAge/kg_formation/processed_data/stitch/stitch_DROSO_CHEMICALENTITY_GENE.csv'

In [48]:
OUTPUT_PATH

'/storage/Arushi/090526_EvoAge/kg_formation/processed_data/stitch/stitch_DROSO_CHEMICALENTITY_GENE.csv'

---
## 1. Load gProfiler Mapping (FlyBase Protein ID → Gene Symbol + Description)

gProfiler maps FlyBase protein IDs (e.g., `FBpp0070873`) to gene symbols (via `converted_alias`) and descriptions.

In [35]:
# # =============================================================================
# # Load gProfiler data and build mapping dictionaries
# # =============================================================================

gprofiler_mapping = pd.read_csv(GPROFILER_PATH)
gprofiler_mapping = gprofiler_mapping[~gprofiler_mapping['converted_alias'].isna()]

# Dictionary: FlyBase Protein ID → Gene Symbol
protein_to_gene_dict = dict(zip(gprofiler_mapping['initial_alias'], gprofiler_mapping['converted_alias']))

# Dictionary: Gene Symbol → Gene Description
gene_to_description_dict = dict(zip(gprofiler_mapping['converted_alias'], gprofiler_mapping['description']))

print(f"gProfiler entries loaded: {len(gprofiler_mapping):,}")
print(f"Protein → Gene dictionary size: {len(protein_to_gene_dict):,}")
print(f"Gene → Description dictionary size: {len(gene_to_description_dict):,}")

gProfiler entries loaded: 10,663
Protein → Gene dictionary size: 10,663
Gene → Description dictionary size: 10,663


---
## 2. Load PubChem (CID → IUPAC Name)

Load PubChem compound data to build a dictionary for annotating chemical entities with human-readable IUPAC names.

In [9]:
# =============================================================================
# Load PubChem compound data and build CID → IUPAC name dictionary
# =============================================================================

pubchem_df = pd.read_pickle(PUBCHEM_PATH)

# Dictionary: PubChem CID → IUPAC Name (for head_detail_name)
pubchem_cid_to_iupac = dict(zip(pubchem_df['PUBCHEM_COMPOUND_CID'], pubchem_df['PUBCHEM_IUPAC_NAME']))

print(f"PubChem compounds loaded: {len(pubchem_df):,}")
print(f"CID → IUPAC dictionary size: {len(pubchem_cid_to_iupac):,}")

# Free memory
del pubchem_df

PubChem compounds loaded: 119,177,440
CID → IUPAC dictionary size: 119,177,440


---
## 3. Load and Process STITCH Chemical–Protein Data

Load the raw STITCH file, clean chemical and protein IDs, map proteins to gene symbols, and annotate with descriptions.

**Processing steps:**
1. Load raw TSV, rename columns to head/Tail
2. Strip species prefix (`7227.`) from protein IDs
3. Clean chemical IDs (strip STITCH `CIDs`/`CIDm` prefixes and leading zeros)
4. Uppercase protein IDs (required for gProfiler dictionary matching)
5. Add KG metadata and map proteins to genes
6. Map descriptions (PubChem IUPAC for head, gProfiler for tail)

In [36]:
# =============================================================================
# Load raw STITCH chemical-protein interaction file
# =============================================================================

chem_gene_df = pd.read_csv(STITCH_RAW_PATH, sep='\t')
chem_gene_df = chem_gene_df.rename(columns={'chemical': 'head', 'protein': 'Tail'})

# Strip species prefix '7227.' from protein IDs
chem_gene_df['Tail'] = chem_gene_df['Tail'].str.replace('7227.', '', regex=False)

print(f"Raw STITCH interactions loaded: {len(chem_gene_df):,}")

Raw STITCH interactions loaded: 20,087,229


In [37]:
chem_gene_df

,head,Tail,experimental,prediction,database,textmining,combined_score
0,CIDm91758408,FBpp0070873,0,0,0,174,174
1,CIDm91758408,FBpp0071971,0,0,0,167,167
2,CIDm91758408,FBpp0073484,0,0,0,257,257
3,CIDm91758408,FBpp0073990,0,0,0,374,374
4,CIDm91758408,FBpp0076076,0,0,0,167,167
...,...,...,...,...,...,...,...
20087224,CIDs00000001,FBpp0305499,0,0,434,0,434
20087225,CIDs00000001,FBpp0305739,0,0,0,150,150
20087226,CIDs00000001,FBpp0305775,0,0,0,247,247
20087227,CIDs00000001,FBpp0305828,0,0,0,192,192


In [38]:
# =============================================================================
# Clean chemical IDs and add KG metadata
# =============================================================================

def clean_chemical_id(chem_id):
    """Remove STITCH-specific prefixes and leading zeros from chemical IDs."""
    return chem_id.lstrip('CIDs').lstrip('CIDm').lstrip('0')

# Note: DataFrame.applymap() is deprecated in newer pandas — should use .map() instead
chem_gene_df[['head']] = chem_gene_df[['head']].applymap(clean_chemical_id)

# Add KG metadata
chem_gene_df['relation'] = 'ChemicalEntity_Gene'
chem_gene_df['head_type'] = 'ChemicalEntity'
chem_gene_df['tail_type'] = 'Gene'

print(f"Chemical IDs cleaned")
chem_gene_df

/tmp/ipykernel_1646139/3545766588.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  chem_gene_df[['head']] = chem_gene_df[['head']].applymap(clean_chemical_id)


Chemical IDs cleaned


,head,Tail,experimental,prediction,database,textmining,combined_score,relation,head_type,tail_type
0,91758408,FBpp0070873,0,0,0,174,174,ChemicalEntity_Gene,ChemicalEntity,Gene
1,91758408,FBpp0071971,0,0,0,167,167,ChemicalEntity_Gene,ChemicalEntity,Gene
2,91758408,FBpp0073484,0,0,0,257,257,ChemicalEntity_Gene,ChemicalEntity,Gene
3,91758408,FBpp0073990,0,0,0,374,374,ChemicalEntity_Gene,ChemicalEntity,Gene
4,91758408,FBpp0076076,0,0,0,167,167,ChemicalEntity_Gene,ChemicalEntity,Gene
...,...,...,...,...,...,...,...,...,...,...
20087224,1,FBpp0305499,0,0,434,0,434,ChemicalEntity_Gene,ChemicalEntity,Gene
20087225,1,FBpp0305739,0,0,0,150,150,ChemicalEntity_Gene,ChemicalEntity,Gene
20087226,1,FBpp0305775,0,0,0,247,247,ChemicalEntity_Gene,ChemicalEntity,Gene
20087227,1,FBpp0305828,0,0,0,192,192,ChemicalEntity_Gene,ChemicalEntity,Gene


In [39]:
# =============================================================================
# Uppercase protein IDs and map to gene symbols
# =============================================================================

# Uppercase protein IDs (required for gProfiler dictionary matching)
chem_gene_df['Tail'] = chem_gene_df['Tail'].str.upper()

# Add remaining metadata
chem_gene_df['relation_type'] = np.nan
chem_gene_df['kg_source'] = 'STITCH'

# Map PubChem IUPAC names to chemical IDs (head_detail_name)
chem_gene_df['head_detail_name'] = chem_gene_df['head'].astype(str).map(pubchem_cid_to_iupac)

# Map FlyBase Protein IDs to gene symbols using gProfiler
chem_gene_df['tail'] = chem_gene_df['Tail'].map(protein_to_gene_dict)

# Set identifier namespaces
chem_gene_df['head_id_is'] = 'Pubchem'
chem_gene_df['tail_id_is'] = 'Flybase'

print(f"Protein IDs mapped to gene symbols")
print(f"Mapped genes: {chem_gene_df['tail'].notna().sum():,}")
print(f"Unmapped (NaN): {chem_gene_df['tail'].isna().sum():,}")

Protein IDs mapped to gene symbols
Mapped genes: 19,125,989
Unmapped (NaN): 961,240


In [40]:
# =============================================================================
# Map gene descriptions
# =============================================================================

chem_gene_df['tail_detail_name'] = chem_gene_df['tail'].map(gene_to_description_dict)

print(f"Descriptions mapped")
print(f"Tail descriptions: {chem_gene_df['tail_detail_name'].notna().sum():,}")
print(f"Head descriptions: {chem_gene_df['head_detail_name'].notna().sum():,}")

Descriptions mapped
Tail descriptions: 13,644,175
Head descriptions: 19,399,822


---
## 4. Filter, Validate, Deduplicate & Export

Filter out entries with unmapped genes/descriptions, add species, validate, deduplicate, and export.

In [41]:
# =============================================================================
# Filter out entries with unmapped entities
# =============================================================================

before_count = len(chem_gene_df)
chem_gene_df = chem_gene_df[~chem_gene_df['tail'].isna()]
chem_gene_df = chem_gene_df[~chem_gene_df['tail_detail_name'].isna()]
chem_gene_df = chem_gene_df[~chem_gene_df['head_detail_name'].isna()]
after_count = len(chem_gene_df)

print(f"Before filtering: {before_count:,}")
print(f"After filtering: {after_count:,}")
print(f"Removed: {before_count - after_count:,} triples with unmapped entities")

Before filtering: 20,087,229
After filtering: 13,203,915
Removed: 6,883,314 triples with unmapped entities


In [42]:
# =============================================================================
# Add species metadata
# =============================================================================
# Note: SettingWithCopyWarning may appear here; it does not affect correctness

chem_gene_df['species'] = 'D.melanogaster'

print(f"Final triples: {len(chem_gene_df):,}")

Final triples: 13,203,915


In [43]:
# =============================================================================
# Data quality validation
# =============================================================================

true_nan_count = chem_gene_df.isna().sum()
string_nan_count = chem_gene_df.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

nan_summary = pd.DataFrame({
    'NaN_count': true_nan_count,
    "'NAN'_string_count": string_nan_count,
    'Total_NaN_like': true_nan_count + string_nan_count
})

print("NaN validation summary:")
nan_summary

NaN validation summary:


,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
Tail,0,0,0
experimental,0,0,0
prediction,0,0,0
database,0,0,0
textmining,0,0,0
combined_score,0,0,0
relation,0,0,0
head_type,0,0,0
tail_type,0,0,0


In [44]:
# =============================================================================
# Deduplicate by grouping on (head, relation, tail)
# =============================================================================

group_cols = ['head', 'relation', 'tail']

def merge_sources(x):
    """Merge multiple kg_source values with '::' separator."""
    return '::'.join(sorted(set(x.dropna())))

chem_gene_dedup = chem_gene_df.groupby(group_cols, as_index=False).agg({
    'head_type': 'first',
    'relation_type': 'first',
    'tail_type': 'first',
    'kg_source': merge_sources,
    'head_id_is': 'first',
    'tail_id_is': 'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'species': 'first'
})

print(f"Triples before deduplication: {len(chem_gene_df):,}")
print(f"Triples after deduplication: {len(chem_gene_dedup):,}")
print(f"Duplicates removed: {len(chem_gene_df) - len(chem_gene_dedup):,}")
chem_gene_dedup.head()

Triples before deduplication: 13,203,915
Triples after deduplication: 7,661,269
Duplicates removed: 5,542,646


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
0,1,ChemicalEntity_Gene,18w,ChemicalEntity,NaN,Gene,STITCH,Pubchem,Flybase,3-acetyloxy-4-(trimethylazaniumyl)butanoate,18 wheeler,D.melanogaster
1,1,ChemicalEntity_Gene,2mit,ChemicalEntity,NaN,Gene,STITCH,Pubchem,Flybase,3-acetyloxy-4-(trimethylazaniumyl)butanoate,2mit,D.melanogaster
2,1,ChemicalEntity_Gene,ATPCL,ChemicalEntity,NaN,Gene,STITCH,Pubchem,Flybase,3-acetyloxy-4-(trimethylazaniumyl)butanoate,ATP citrate lyase,D.melanogaster
3,1,ChemicalEntity_Gene,ATPsynbeta,ChemicalEntity,NaN,Gene,STITCH,Pubchem,Flybase,3-acetyloxy-4-(trimethylazaniumyl)butanoate,"ATP synthase, beta subunit",D.melanogaster
4,1,ChemicalEntity_Gene,ATPsynbetaL,ChemicalEntity,NaN,Gene,STITCH,Pubchem,Flybase,3-acetyloxy-4-(trimethylazaniumyl)butanoate,"ATP synthase, beta subunit-like",D.melanogaster


In [49]:
# =============================================================================
# Export final deduplicated Chemical-Gene KG triples
# =============================================================================

chem_gene_dedup.to_csv(OUTPUT_PATH, index=None)
print(f"Final output saved to: {OUTPUT_PATH}")
print(f"Total triples exported: {len(chem_gene_dedup):,}")

Final output saved to: /storage/Arushi/090526_EvoAge/kg_formation/processed_data/stitch/stitch_DROSO_CHEMICALENTITY_GENE.csv
Total triples exported: 7,661,269


---
## Summary

This notebook processed STITCH Chemical–Protein interaction data for *D. melanogaster* into standardized Chemical–Gene KG triples:

1. **gProfiler setup** — Loaded FlyBase protein → gene symbol mappings and descriptions
2. **PubChem setup** — Loaded CID → IUPAC name dictionary
3. **STITCH processing** — Loaded ~20M raw interactions, cleaned chemical/protein IDs, mapped to genes
4. **Annotation & filtering** — Added descriptions, filtered unmapped (~20M → ~13.6M)
5. **Deduplication** — Grouped by (head, relation, tail) for ~7.7M unique final triples

### Output

| File | Description |
|------|-------------|
| `stitch_DROSO_CHEMICALENTITY_GENE.csv` | Final: deduplicated relation-wise KG triples |